# 02: Schedule Variance Analysis

**Goal:** Analyze federal IT project schedule variance patterns from ITDashboard.gov to understand which projects are slipping and why.

**Data:** ITDashboard.gov project performance data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## Load Data

In [ ]:
df = pd.read_csv('../data/itdashboard_projects.csv')
print(f"Total projects: {len(df)}")
print(f"Agencies: {df['agency_code'].nunique()}")
df.head()

## Schedule Variance Distribution

In [ ]:
df['schedule_variance_days'] = pd.to_numeric(df['schedule_variance_days'], errors='coerce').fillna(0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
df['schedule_variance_days'].hist(bins=30, ax=axes[0], color='#3498db', edgecolor='white')
axes[0].axvline(df['schedule_variance_days'].mean(), color='red', linestyle='--', label=f"Mean: {df['schedule_variance_days'].mean():.0f}d")
axes[0].set_title('Schedule Variance Distribution')
axes[0].set_xlabel('Days')
axes[0].legend()

# By health
for health, color in [('Green', '#2ecc71'), ('Yellow', '#f1c40f'), ('Red', '#e74c3c')]:
    subset = df[df['project_health'] == health]
    if len(subset) > 0:
        subset['schedule_variance_days'].hist(bins=20, alpha=0.6, ax=axes[1], label=health, color=color)
axes[1].set_title('Schedule Variance by Project Health')
axes[1].set_xlabel('Days')
axes[1].legend()

plt.tight_layout()
plt.show()

## Agency-Level Analysis

In [ ]:
agency_stats = df.groupby('agency_code').agg({
    'schedule_variance_days': ['mean', 'std', 'count'],
    'project_health': lambda x: (x == 'Red').mean()
}).round(2)
agency_stats.columns = ['avg_delay', 'delay_std', 'project_count', 'red_rate']
agency_stats = agency_stats.sort_values('avg_delay', ascending=False)

agency_stats.head(10)

In [ ]:
# Visualize
fig, ax = plt.subplots(figsize=(12, 6))
agency_stats.head(10)['avg_delay'].plot(kind='barh', color='#e67e22')
plt.title('Average Schedule Delay by Agency')
plt.xlabel('Days')
plt.show()

## Compute SPI Estimates

In [ ]:
# Estimate SPI from schedule variance
df['estimated_planned_duration'] = 365
df['spi_estimate'] = np.where(
    df['estimated_planned_duration'] > 0,
    1 - (df['schedule_variance_days'] / df['estimated_planned_duration']),
    1.0
)
df['spi_estimate'] = df['spi_estimate'].clip(lower=0.5, upper=1.0)

print(f"Mean SPI: {df['spi_estimate'].mean():.3f}")
print(f"SPI < 0.9 (at risk): {(df['spi_estimate'] < 0.9).sum()} projects")
print(f"SPI < 0.8 (critical): {(df['spi_estimate'] < 0.8).sum()} projects")

In [ ]:
# Save enhanced data
df.to_csv('../data/projects_schedule_enhanced.csv', index=False)
print("Enhanced data saved.")